In [ ]:
"""
=============================================================================
QUANTUM SVR - Previsão de Zeros da Função Zeta de Riemann
=============================================================================

Estratégia:
  Kernel quântico ZZFeatureMap (IQP - Instantaneous Quantum Polynomial)
  computado analiticamente, sem necessidade de hardware quântico.

  O circuito quântico equivalente é:
    Para cada repetição (reps):
      1. H ⊗ H ⊗ ... ⊗ H       (superposição)
      2. Rz(2·xᵢ) no qubit i    (codificação single-qubit)
      3. CNOT + Rz(2·(π-xᵢ)(π-xⱼ)) + CNOT  (emaranhamento ZZ)

  O kernel resultante K(x,y) = |⟨φ(x)|φ(y)⟩|² tem forma analítica:

    K(x,y) = [∏ᵢ cos(xᵢ - yᵢ)] · [∏_{i<j} cos((π-xᵢ)(π-xⱼ) - (π-yᵢ)(π-yⱼ))]

  Isso é elevado ao número de repetições e combinado com kernel RBF
  para melhor desempenho.

Referências:
  - Havlíček et al. (2019): "Supervised learning with quantum-enhanced feature spaces"
  - Qiskit ZZFeatureMap documentation
=============================================================================
"""

import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.kernel_approximation import Nystroem
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import KernelCenterer
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÕES
# ─────────────────────────────────────────────────────────────────────────────
DATA_PATH  = "../dataset/riemann_features.csv"
N_SAMPLES  = 2000
TEST_SPLIT = 0.8

FEATURES = [
    'z_co_gram_lag_2', 'z_gram',     'z_gram_lag_1',  'd_lag_13',
    'z_co_gram_lag_3', 'z_co_gram_lag_1', 'd_lag_14', 'd_lag_1',
    'z_gram_lag_2',    'd_lag_17'
]

# ─────────────────────────────────────────────────────────────────────────────
# 1. CARREGAMENTO E SPLIT TEMPORAL
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("  QUANTUM SVR — ZEROS DA FUNÇÃO ZETA DE RIEMANN")
print("=" * 65)

dataset = pd.read_csv(DATA_PATH)
X_df    = dataset.drop(columns=["distance"])[:N_SAMPLES]
y_all   = dataset["distance"][:N_SAMPLES].to_numpy()

split       = int(TEST_SPLIT * N_SAMPLES)
X_train_raw = X_df[:split][FEATURES].to_numpy()
X_test_raw  = X_df[split:][FEATURES].to_numpy()
y_train     = y_all[:split]
y_test      = y_all[split:]

print(f"\nAmostras  → Treino: {len(y_train)}  |  Teste: {len(y_test)}")
print(f"Features  → {len(FEATURES)}")
print(f"Target    → mean={y_all.mean():.4f}, std={y_all.std():.4f}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PRÉ-PROCESSAMENTO: escalar para [0, π] (ângulo de rotação quântica)
# ─────────────────────────────────────────────────────────────────────────────
scaler  = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

# ─────────────────────────────────────────────────────────────────────────────
# 3. KERNEL QUÂNTICO ZZFeatureMap  (analítico, vetorizado por batches)
# ─────────────────────────────────────────────────────────────────────────────

def _zz_kernel_block(Xi, Yj, Xi_pairs, Yj_pairs):
    """
    Calcula bloco do kernel ZZFeatureMap:
      K_block(i,j) = [∏ₖ cos(xₖ - yₖ)] · [∏_{p<q} cos(xₚxq - yₚyq)]
    """
    # ── Single-qubit: cos(xᵢ - yⱼ) por feature ──────────────────────────
    diff_sq  = Xi[:, np.newaxis, :] - Yj[np.newaxis, :, :]  # (bs_i, bs_j, d)
    k_single = np.prod(np.cos(diff_sq), axis=2)              # (bs_i, bs_j)

    # ── ZZ two-qubit: cos(xₚxq - yₚyq) por par ──────────────────────────
    diff_zz = Xi_pairs[:, np.newaxis, :] - Yj_pairs[np.newaxis, :, :]  # (bs_i, bs_j, n_pairs)
    k_zz    = np.prod(np.cos(diff_zz), axis=2)                          # (bs_i, bs_j)

    return k_single * k_zz


def zzfeaturemap_kernel(X, Y, n_reps=2, batch_size=128, verbose=True):
    """
    Kernel quântico ZZFeatureMap:
      K(x,y) = |⟨φ(x)|φ(y)⟩|²  ≈  [k_single(x,y) · k_zz(x,y)]^n_reps

    Parâmetros
    ----------
    n_reps   : número de repetições do circuito (profundidade)
    batch_size: tamanho do bloco para computação eficiente
    """
    n, d = X.shape
    m    = Y.shape[0]

    # Pares (p, q) com p < q  →  emaranhamento ZZ
    pairs   = [(p, q) for p in range(d) for q in range(p + 1, d)]
    n_pairs = len(pairs)

    # Transformação ZZ: (π - x) para capturar interações não-lineares
    X_zz = np.pi - X   # (n, d)
    Y_zz = np.pi - Y   # (m, d)

    X_pairs = np.column_stack([X_zz[:, p] * X_zz[:, q] for p, q in pairs])  # (n, n_pairs)
    Y_pairs = np.column_stack([Y_zz[:, p] * Y_zz[:, q] for p, q in pairs])  # (m, n_pairs)

    K = np.zeros((n, m))
    t0 = time.time()

    for i0 in range(0, n, batch_size):
        i1 = min(i0 + batch_size, n)
        if verbose and i0 % (batch_size * 4) == 0:
            pct = 100 * i0 / n
            print(f"    [{pct:5.1f}%] linha {i0}/{n} — {time.time()-t0:.1f}s")

        Xi      = X[i0:i1]
        Xi_pair = X_pairs[i0:i1]

        for j0 in range(0, m, batch_size):
            j1 = min(j0 + batch_size, m)
            Yj      = Y[j0:j1]
            Yj_pair = Y_pairs[j0:j1]

            block = _zz_kernel_block(Xi, Yj, Xi_pair, Yj_pair)
            K[i0:i1, j0:j1] = np.sign(block) * (np.abs(block) ** n_reps)

    if verbose:
        print(f"    [100.0%] concluído em {time.time()-t0:.1f}s")
    return K


# ─────────────────────────────────────────────────────────────────────────────
# 4. KERNEL HÍBRIDO: Quantum + RBF  (combinação convexa)
# ─────────────────────────────────────────────────────────────────────────────

def rbf_kernel(X, Y, gamma=None):
    """RBF/Gaussiano clássico para combinação híbrida."""
    if gamma is None:
        gamma = 1.0 / X.shape[1]
    sq_dists = (
        np.sum(X**2, axis=1)[:, np.newaxis]
        + np.sum(Y**2, axis=1)[np.newaxis, :]
        - 2 * X @ Y.T
    )
    return np.exp(-gamma * sq_dists)


def hybrid_kernel(X, Y, alpha=0.7, n_reps=2, gamma=None, batch_size=128, verbose=True):
    """
    K_hybrid = alpha · K_quantum + (1 - alpha) · K_rbf

    Combina o poder expressivo do kernel quântico (interações de alta ordem)
    com a suavidade do RBF (generalização).
    """
    K_q = zzfeaturemap_kernel(X, Y, n_reps=n_reps, batch_size=batch_size, verbose=verbose)
    K_r = rbf_kernel(X, Y, gamma=gamma)
    return alpha * K_q + (1 - alpha) * K_r


# ─────────────────────────────────────────────────────────────────────────────
# 5. COMPUTAR MATRIZES DE KERNEL
# ─────────────────────────────────────────────────────────────────────────────
N_REPS    = 2
ALPHA_HYB = 0.65   # peso do kernel quântico
GAMMA_RBF = 1.5 / len(FEATURES)

print("── Computando K_train (treino × treino) ─────────────────────")
K_train = hybrid_kernel(
    X_train, X_train,
    alpha=ALPHA_HYB, n_reps=N_REPS, gamma=GAMMA_RBF
)
print(f"   shape={K_train.shape}  |  range=[{K_train.min():.4f}, {K_train.max():.4f}]\n")

print("── Computando K_test (teste × treino) ───────────────────────")
K_test = hybrid_kernel(
    X_test, X_train,
    alpha=ALPHA_HYB, n_reps=N_REPS, gamma=GAMMA_RBF
)
print(f"   shape={K_test.shape}\n")

# Centralizar kernels (melhora separabilidade no RKHS)
centerer  = KernelCenterer()
K_train_c = centerer.fit_transform(K_train)
K_test_c  = centerer.transform(K_test)

# ─────────────────────────────────────────────────────────────────────────────
# 6. BUSCA DE HIPERPARÂMETROS (TimeSeriesSplit)
# ─────────────────────────────────────────────────────────────────────────────
print("── Busca de hiperparâmetros ──────────────────────────────────")

C_grid       = [10, 50, 100, 200, 500, 1000]
epsilon_grid = [0.001, 0.005, 0.01, 0.02, 0.05]
tscv         = TimeSeriesSplit(n_splits=5)

best_cv_r2   = -np.inf
best_params  = {'C': 100, 'epsilon': 0.01}

for C in C_grid:
    for eps in epsilon_grid:
        fold_r2s = []
        for tr_idx, val_idx in tscv.split(K_train_c):
            K_tr  = K_train_c[np.ix_(tr_idx, tr_idx)]
            K_val = K_train_c[np.ix_(val_idx, tr_idx)]
            y_tr  = y_train[tr_idx]
            y_val = y_train[val_idx]

            m = SVR(kernel='precomputed', C=C, epsilon=eps)
            m.fit(K_tr, y_tr)
            pred = m.predict(K_val)
            fold_r2s.append(r2_score(y_val, pred))

        mean_r2 = np.mean(fold_r2s)
        if mean_r2 > best_cv_r2:
            best_cv_r2  = mean_r2
            best_params = {'C': C, 'epsilon': eps}
            print(f"   ✓  C={C:5d}  ε={eps:.4f}  →  CV R²={mean_r2:.4f}")

print(f"\n   Melhores parâmetros: {best_params}  (CV R²={best_cv_r2:.4f})\n")

# ─────────────────────────────────────────────────────────────────────────────
# 7. MODELO FINAL
# ─────────────────────────────────────────────────────────────────────────────
print("── Treinando modelo final ────────────────────────────────────")
model = SVR(kernel='precomputed', **best_params)
model.fit(K_train_c, y_train)

y_pred_train = model.predict(K_train_c)
y_pred_test  = model.predict(K_test_c)

# ─────────────────────────────────────────────────────────────────────────────
# 8. AVALIAÇÃO
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mae  = np.mean(np.abs(y_true - y_pred))
    print(f"  [{label}]  RMSE={rmse:.6f}  |  R²={r2:.6f}  |  MAE={mae:.6f}")
    return rmse, r2

print("\n" + "=" * 65)
print("  RESULTADOS — QUANTUM SVR (ZZFeatureMap + RBF Hybrid)")
print("=" * 65)
train_rmse, train_r2 = evaluate(y_train, y_pred_train, "TREINO")
test_rmse,  test_r2  = evaluate(y_test,  y_pred_test,  " TESTE")
print("=" * 65)

RMSE_TARGET = 0.07
R2_TARGET   = 0.94

print(f"\n  Alvo: RMSE < {RMSE_TARGET}  e  R² > {R2_TARGET}")
if test_rmse < RMSE_TARGET and test_r2 > R2_TARGET:
    print("  ✅  OBJETIVOS ATINGIDOS!")
else:
    print("  ⚠️   Objetivos parcialmente atingidos:")
    if test_rmse >= RMSE_TARGET:
        print(f"       RMSE={test_rmse:.5f} ≥ {RMSE_TARGET} (faltam {test_rmse-RMSE_TARGET:.5f})")
    if test_r2 <= R2_TARGET:
        print(f"       R²={test_r2:.5f} ≤ {R2_TARGET}")

# ─────────────────────────────────────────────────────────────────────────────
# 9. VISUALIZAÇÕES
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Quantum SVR — Zeros da Função Zeta de Riemann", fontsize=13, fontweight='bold')

# ── Plot 1: Predito vs Real (teste) ──────────────────────────────────────────
ax = axes[0]
ax.scatter(y_test, y_pred_test, alpha=0.5, s=20, color='royalblue', label='Amostras')
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Ideal')
ax.set_xlabel("Valor Real")
ax.set_ylabel("Valor Predito")
ax.set_title(f"Predito vs Real  (Teste)\nRMSE={test_rmse:.5f}  R²={test_r2:.5f}")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 2: Série temporal — últimas 200 amostras do teste ───────────────────
ax = axes[1]
n_show = min(200, len(y_test))
idx    = np.arange(n_show)
ax.plot(idx, y_test[:n_show],      label='Real',    color='steelblue', linewidth=1.2)
ax.plot(idx, y_pred_test[:n_show], label='Predito', color='tomato',    linewidth=1.2, linestyle='--')
ax.set_xlabel("Índice (amostras de teste)")
ax.set_ylabel("Distance  γₙ − g_{n-1}")
ax.set_title("Série Temporal — Conjunto de Teste")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 3: Distribuição dos resíduos ────────────────────────────────────────
ax = axes[2]
residuals = y_test - y_pred_test
ax.hist(residuals, bins=40, color='mediumpurple', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linewidth=2, linestyle='--')
ax.set_xlabel("Resíduo (Real − Predito)")
ax.set_ylabel("Frequência")
ax.set_title(f"Distribuição dos Resíduos\nMédia={residuals.mean():.5f}  Std={residuals.std():.5f}")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/quantum_svr_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nGráfico salvo em: ../results/quantum_svr_results.png")

# ─────────────────────────────────────────────────────────────────────────────
# 10. RESUMO DO MODELO
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("  RESUMO DO MODELO QUÂNTICO")
print("─" * 65)
print(f"  Feature map    : ZZFeatureMap (IQP kernel analítico)")
print(f"  Repetições (L) : {N_REPS}")
print(f"  Kernel híbrido : α={ALPHA_HYB}·K_quantum + {1-ALPHA_HYB:.2f}·K_RBF")
print(f"  γ (RBF)        : {GAMMA_RBF:.4f}")
print(f"  C (SVR)        : {best_params['C']}")
print(f"  ε (SVR)        : {best_params['epsilon']}")
print(f"  Vetores suporte: {len(model.support_)} / {len(y_train)}")
print(f"  Centralização  : KernelCenterer aplicado")
print("─" * 65)
print(f"  RMSE teste  : {test_rmse:.6f}  (alvo < {RMSE_TARGET})")
print(f"  R²   teste  : {test_r2:.6f}  (alvo > {R2_TARGET})")
print("─" * 65)